# 01 — Preprocessing

Explore liver CT intensity windowing, classical denoising, and cropping before the later segmentation work. These examples rely on the provided liver reference mask to define the crop and do not implement segmentation.

## 1. Load a selected slice and inspect HU values
Use one case at a time. Review the configured liver window (default center 60 HU, width 400 HU) and adjust it if the displayed anatomy warrants a different choice.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from psib.config import load_config
from psib.data_io import get_slice, load_case
from psib.preprocessing import apply_hu_window, preprocess_slice
cfg = load_config(ROOT / 'config.yaml')
chosen = pd.read_csv(Path(cfg['_output_dir']) / 'selected_cases_slices.csv')
row = chosen.iloc[0]
ct, seg, spacing = load_case(int(row.case_id), cfg['data_dir'])
slice_hu = get_slice(ct, int(row.slice_index))
liver_mask = get_slice(seg, int(row.slice_index)) > 0
spacing

## 2. Compare denoising filters
Compare median, Gaussian, and bilateral filters visually on the same windowed slice. A simple noise summary such as standard deviation in a selected homogeneous region can support discussion, but it is not a segmentation-accuracy metric.

In [ ]:
from psib.preprocessing import denoise_bilateral, denoise_gaussian, denoise_median
windowed = apply_hu_window(slice_hu, cfg['preprocessing']['hu_window']['center'], cfg['preprocessing']['hu_window']['width'])
median_image = denoise_median(windowed, size=3)
gaussian_image = denoise_gaussian(windowed, sigma=1.0)
bilateral_image = denoise_bilateral(windowed, sigma_color=0.08, sigma_spatial=3.0)

## 3. Crop around the liver and preserve coordinates
The returned crop box is `(x0, y0, x1, y1)` in the full slice. Add its origin back to crop-local coordinates when mapping downstream points or masks to the original image.

In [ ]:
processed = preprocess_slice(slice_hu, liver_mask, cfg)
processed['crop_box'], processed['windowed'].shape, processed['cropped'].shape

## Discussion notes
Record the chosen HU window, the filter and parameters, any optional equalization, and how noise reduction affected tumor/liver boundary visibility. Include before/after figures from `results/figures/` and state that the crop uses the reference liver label.